<a href="https://colab.research.google.com/github/DCI-alxogm/me2025-clase-Rubi200415/blob/main/Ajuste_minimos_cuadrados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [48]:
uploaded = files.upload()

Saving Experimental water adsorption isotherms (2) (1).xlsx to Experimental water adsorption isotherms (2) (1).xlsx


In [49]:
datos_experimentales = pd.read_excel(list(uploaded.keys())[0])

In [50]:
granos_tostados = datos_experimentales[datos_experimentales["Type"] == 'Roasted beans']
granos_replica1 = granos_tostados[granos_tostados["Replicate"] == 1]
gt_25 = granos_replica1[granos_replica1["Temperature"] == 25]
gt_35 = granos_replica1[granos_replica1["Temperature"] == 35]
gt_45 = granos_replica1[granos_replica1["Temperature"] == 45]

In [51]:
print("Datos cargados correctamente!")
print(f"Total de registros: {len(datos_experimentales)}")
print(f"Granos tostados - Réplica 1: {len(granos_replica1)}")

Datos cargados correctamente!
Total de registros: 1860
Granos tostados - Réplica 1: 153


In [52]:
import pandas as pd
import os
import subprocess

# --- CONFIGURACIÓN ---
EXCEL_FILE = "Experimental water adsorption isotherms (2).xlsx"
OUTPUT_DIR = "curvas_csv"
C_EXECUTABLE = "./ajuste"  # Cambia si tu ejecutable tiene otro nombre o ruta

# Crear directorio de salida
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cargar y limpiar datos
df = pd.read_excel(EXCEL_FILE, skiprows=1)
df.columns = [
    "Type", "Replicate", "Temperature", "Water activity",
    "Moisture content (% wet basis)", "Moisture content (% dry basis)"
]

# Tipos y temperaturas de interés
types = ["Roasted beans", "Ground-Medium", "Ground-Fine"]
temps = [25, 35, 45]

# Lista para resumen final
outputs = []

for typ in types:
    for T in temps:
        # Filtrar datos
        subset = df[(df["Type"] == typ) & (df["Temperature"] == T)]
        if subset.empty:
            print(f"⚠️  Sin datos para {typ} a {T}°C")
            continue

        # Seleccionar solo columnas necesarias y eliminar filas inválidas
        subset_clean = subset[["Water activity", "Moisture content (% dry basis)"]].copy()
        subset_clean = subset_clean[
            (subset_clean["Water activity"] > 0) &
            (subset_clean["Water activity"] < 1)
        ]

        if len(subset_clean) < 4:
            print(f" Pocos puntos para {typ} a {T}°C. Saltando.")
            continue

        # Nombre del archivo CSV
        safe_type = typ.replace(" ", "_").replace("-", "_")
        csv_filename = f"{OUTPUT_DIR}/{safe_type}_{T}C.csv"
        subset_clean.to_csv(csv_filename, index=False, header=False)
        print(f" Guardado: {csv_filename}")

        # Opcional: ejecutar el programa en C
        if os.path.exists(C_EXECUTABLE):
            print(f"▶ Ejecutando ajuste en C para {typ} a {T}°C...")
            result = subprocess.run([C_EXECUTABLE, csv_filename], capture_output=True, text=True)
            if result.returncode == 0:
                print(result.stdout)
                outputs.append(f"\n=== {typ} {T}°C ===\n{result.stdout}")
            else:
                print("Error al ejecutar el programa en C:")
                print(result.stderr)

# Guardar resumen de resultados en C (opcional)
if outputs:
    with open(f"{OUTPUT_DIR}/resultados_c_ajustes.txt", "w") as f:
        f.write("".join(outputs))
    print(f" Resultados guardados en {OUTPUT_DIR}/resultados_c_ajustes.txt")

 Guardado: curvas_csv/Roasted_beans_25C.csv
▶ Ejecutando ajuste en C para Roasted beans a 25°C...

--- Ajuste Peleg ---
Iteraciones: 441
Chi2 final: 39.749682
Parámetros: [1168.732128, -47.149331, 2.745263, 0.164296]

--- Ajuste DLP ---
Iteraciones: 193
Chi2 final: 101.629734
Parámetros: [2.986157, 0.415233, -1.766698, -1.330450]

 Guardado: curvas_csv/Roasted_beans_35C.csv
▶ Ejecutando ajuste en C para Roasted beans a 35°C...

--- Ajuste Peleg ---
Iteraciones: 215
Chi2 final: 22.665775
Parámetros: [27.735521, -12.857849, 2.381338, 0.086471]

--- Ajuste DLP ---
Iteraciones: 147
Chi2 final: 23.629008
Parámetros: [2.279094, 0.249829, -0.278708, -0.809055]

 Guardado: curvas_csv/Roasted_beans_45C.csv
▶ Ejecutando ajuste en C para Roasted beans a 45°C...

--- Ajuste Peleg ---
Iteraciones: 198
Chi2 final: 9.322417
Parámetros: [28.616419, -10.070888, 2.347753, 0.118506]

--- Ajuste DLP ---
Iteraciones: 173
Chi2 final: 10.375384
Parámetros: [2.019405, 0.071381, 0.339014, -0.779058]

 Guardado

In [53]:
!apt-get update -qq
!apt-get install -y -qq gcc libgsl-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [54]:
# Compilar
!gcc -o ajuste ajuste.c -lgsl -lgslcblas -lm

# Ejecutar
!./ajuste

Uso: ./ajuste <archivo.csv>
Formato CSV: aw,xe (una línea por dato, sin encabezado necesario)


In [56]:
%%writefile ajuste.c

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <gsl/gsl_multimin.h>
#include <string.h>   // Para strchr
#include <stddef.h>   // Para size_t

// Función auxiliar para cargar datos desde CSV (mejorada)
int load_data(const char* filename, double** aw, double** xe, size_t* n) {
    FILE* fp = fopen(filename, "r");
    if (!fp) {
        perror("No se pudo abrir el archivo");
        return -1;
    }

    // Contar líneas válidas (ignorar encabezado si existe)
    size_t lines = 0;
    char buffer[256];
    int first_line = 1; // Para saltar encabezado si existe

    while (fgets(buffer, sizeof(buffer), fp)) {
        // Saltar encabezado si es la primera línea y contiene texto (no números)
        if (first_line) {
            first_line = 0;
            // Si la primera línea parece tener letras, la ignoramos
            if (strchr(buffer, 'a') || strchr(buffer, 'A') || strchr(buffer, 'w')) {
                continue;
            }
        }
        // Contar solo líneas con datos numéricos
        lines++;
    }
    rewind(fp);

    // Saltar encabezado si existe
    first_line = 1;
    while (fgets(buffer, sizeof(buffer), fp)) {
        if (first_line) {
            first_line = 0;
            if (strchr(buffer, 'a') || strchr(buffer, 'A') || strchr(buffer, 'w')) {
                continue;
            }
        }
        break;
    }

    *n = lines;
    if (*n == 0) {
        fprintf(stderr, "Error: No se encontraron datos en %s\n", filename);
        fclose(fp);
        return -1;
    }

    *aw = malloc(*n * sizeof(double));
    *xe = malloc(*n * sizeof(double));

    if (*aw == NULL || *xe == NULL) {
        fprintf(stderr, "Error: No se pudo asignar memoria.\n");
        fclose(fp);
        return -1;
    }

    size_t i = 0;
    while (i < *n && fscanf(fp, "%lf,%lf", &(*aw)[i], &(*xe)[i]) == 2) {
        if ((*aw)[i] > 0 && (*aw)[i] < 1) {
            i++;
        }
    }
    *n = i; // Solo puntos válidos

    if (*n < 4) {
        fprintf(stderr, "Advertencia: Pocos puntos válidos (%zu) en %s\n", *n, filename);
        free(*aw);
        free(*xe);
        *aw = NULL;
        *xe = NULL;
        fclose(fp);
        return -1;
    }

    fclose(fp);
    return 0;
}

Overwriting ajuste.c


In [57]:
# Compilar (si aún no lo hiciste)
!gcc -o ajuste ajuste.c -lgsl -lgslcblas -lm

# Ejecutar con uno de tus archivos CSV
!./ajuste curvas_csv/Roasted_beans_25C.csv

/usr/bin/ld: /usr/lib/gcc/x86_64-linux-gnu/11/../../../x86_64-linux-gnu/Scrt1.o: in function `_start':
(.text+0x1b): undefined reference to `main'
collect2: error: ld returned 1 exit status
/bin/bash: line 1: ./ajuste: No such file or directory


In [59]:
import os

# Lista de todos los archivos CSV generados
csv_files = [
    "curvas_csv/Roasted_beans_25C.csv",
    "curvas_csv/Roasted_beans_35C.csv",
    "curvas_csv/Roasted_beans_45C.csv",
    "curvas_csv/Ground_Medium_25C.csv",
    "curvas_csv/Ground_Medium_35C.csv",
    "curvas_csv/Ground_Medium_45C.csv",
    "curvas_csv/Ground_Fine_25C.csv",
    "curvas_csv/Ground_Fine_35C.csv",
    "curvas_csv/Ground_Fine_45C.csv"
]

# Verificar que los archivos existen
existing_files = [f for f in csv_files if os.path.exists(f)]
missing_files = [f for f in csv_files if not os.path.exists(f)]

if missing_files:
    print(" Archivos faltantes (no se generaron o tienen nombre distinto):")
    for f in missing_files:
        print(f"  - {f}")
    print()

# Compilar el programa en C (solo una vez)
print("Compilando programa en C...")
!gcc -o ajuste ajuste.c -lgsl -lgslcblas -lm

# Ejecutar el ajuste para cada archivo
print("\n Ejecutando ajustes...\n")
for csv_file in existing_files:
    print(f"\n{'='*60}")
    print(f"Procesando: {csv_file}")
    print(f"{'='*60}")
    !./ajuste "{csv_file}"

# Guardar todos los resultados en un archivo
with open('resultados_c_ajustes.txt', 'w') as f:
    for csv_file in existing_files:
        result = !./ajuste "{csv_file}"
        f.write(f"\n=== {csv_file} ===\n")
        f.write("\n".join(result) + "\n")

print("\n Resultados guardados en 'resultados_c_ajustes.txt'")

Compilando programa en C...
/usr/bin/ld: /usr/lib/gcc/x86_64-linux-gnu/11/../../../x86_64-linux-gnu/Scrt1.o: in function `_start':
(.text+0x1b): undefined reference to `main'
collect2: error: ld returned 1 exit status

 Ejecutando ajustes...


Procesando: curvas_csv/Roasted_beans_25C.csv
/bin/bash: line 1: ./ajuste: No such file or directory

Procesando: curvas_csv/Roasted_beans_35C.csv
/bin/bash: line 1: ./ajuste: No such file or directory

Procesando: curvas_csv/Roasted_beans_45C.csv
/bin/bash: line 1: ./ajuste: No such file or directory

Procesando: curvas_csv/Ground_Medium_25C.csv
/bin/bash: line 1: ./ajuste: No such file or directory

Procesando: curvas_csv/Ground_Medium_35C.csv
/bin/bash: line 1: ./ajuste: No such file or directory

Procesando: curvas_csv/Ground_Medium_45C.csv
/bin/bash: line 1: ./ajuste: No such file or directory

Procesando: curvas_csv/Ground_Fine_25C.csv
/bin/bash: line 1: ./ajuste: No such file or directory

Procesando: curvas_csv/Ground_Fine_35C.csv
/bin/bas